# Voice Engine — speak any text in your voice

```
Step 1  Prepare your recordings          ✔
Step 2  Text → speech in your voice      (English + Hindi)   ← this notebook, now
Step 3  engine.speak("text", language="hi")
Later   Add more voices
```

Run the cells **from top to bottom**. In Colab, choose *Runtime → Change runtime type → T4 GPU* (needed from Step 2).

> Your recordings are **private**. They are never in GitHub — you bring them in from Google Drive or by uploading.

## Setup — get the code

In [ ]:
import os, subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.chdir("/content")
    if not Path("Voice_engine").exists():
        subprocess.run(["git", "clone", "https://github.com/8919134556/Voice_engine.git"], check=True)
    os.chdir("/content/Voice_engine")
    subprocess.run(["git", "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")                                   # running locally from notebooks/
sys.path.insert(0, os.getcwd())
print("Project folder:", os.getcwd())

## Step 1a — bring in your recordings

Your WAV files must end up in `dataset/speaker_01/`. Use **one** of the two cells below.

**Option A — Google Drive.** Upload your `speaker_01` folder to Drive first, then set `DRIVE_FOLDER`
to where it is (the default matches `My Drive/dataset/speaker_01`).

In [ ]:
# Option A: copy from Google Drive
DRIVE_FOLDER = "/content/drive/MyDrive/dataset/speaker_01"     # <- change if your folder is elsewhere

import glob, shutil
from google.colab import drive
drive.mount("/content/drive")                                   # always /content/drive, never the project folder
os.makedirs("dataset/speaker_01", exist_ok=True)
for f in glob.glob(f"{DRIVE_FOLDER}/*.wav"):
    shutil.copy(f, "dataset/speaker_01/")
print("WAV files in dataset/speaker_01:", len(glob.glob("dataset/speaker_01/*.wav")))

**Option B — upload directly** (files disappear when the Colab session ends). Skip if you used Option A.

In [ ]:
# Option B: upload from your computer (select all your WAV files)
import glob
from google.colab import files
os.makedirs("dataset/speaker_01", exist_ok=True)
for name in files.upload():
    os.replace(name, f"dataset/speaker_01/{name}")
print("WAV files in dataset/speaker_01:", len(glob.glob("dataset/speaker_01/*.wav")))

## Step 1b — check and clean the recordings

For every file: is it readable, not silent, not clipped, long enough?
Then a cleaned copy is saved to `dataset_clean/speaker_01/`: mono, 22050 Hz, silence trimmed at
start/end, and **the same loudness for every clip**. Your original files are never changed.

In [ ]:
!python scripts/step1_prepare_dataset.py --voice speaker_01

## Step 1c — listen: original vs cleaned

The cleaned version should sound the same — just without the silence at the edges and at an even volume.

In [ ]:
from IPython.display import Audio, display
name = "audio_001.wav"
print("Original:"); display(Audio(f"dataset/speaker_01/{name}"))
print("Cleaned:");  display(Audio(f"dataset_clean/speaker_01/{name}"))

### Step 1 done when
- the report says **Usable recordings: 20 of 20** (or close), and
- the cleaned audio sounds like you.

---
# Step 2 — text → speech in your voice

```
dataset_clean/speaker_01/*.wav ─► XTTS-v2 listens once ─► voice profile
text + language (en / hi) ──────────────────────────────┴─► speech in your voice (WAV)
```

**XTTS-v2** is a pretrained multilingual voice-cloning model. It already knows how to speak English and Hindi;
it imitates *your* voice from your cleaned recordings. Use a **GPU** runtime (T4) — on CPU it is very slow.

### Step 2a — install the model library (1–2 minutes)

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-tts.txt"],
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-4000:])
    print(result.stderr[-4000:])
    raise RuntimeError("Install failed - copy the messages above and send them.")
print("coqui-tts installed")

### Step 2b — the model license (your decision)

The XTTS-v2 model weights are under the **Coqui Public Model License (CPML)**: free for **non-commercial** use only.
Read it here: https://coqui.ai/cpml — if you agree, change `False` to `True` and run the cell.

In [ ]:
I_ACCEPT_COQUI_CPML = False      # <- set to True only if you have read and agree to https://coqui.ai/cpml

if I_ACCEPT_COQUI_CPML:
    os.environ["COQUI_TOS_AGREED"] = "1"
    print("License accepted - you can continue.")
else:
    print("Not accepted yet: read https://coqui.ai/cpml and set I_ACCEPT_COQUI_CPML = True to continue.")

### Step 2c — load the model and your voice

First time only: downloads the model (~1.8 GB, 1–3 minutes). Then it listens to your cleaned recordings once.

In [ ]:
import time
from voice_engine.tts import VoiceCloner

start = time.time()
cloner = VoiceCloner()
print(f"Model loaded on {cloner.device} in {time.time() - start:.0f}s")
start = time.time()
n = cloner.load_voice("speaker_01")
print(f"Your voice loaded from {n} recordings in {time.time() - start:.1f}s")

### Step 2d — English in your voice

In [ ]:
from IPython.display import Audio, display
path = cloner.speak("Hello! This is my own voice, generated by my voice engine.", language="en")
print("Saved:", path)
display(Audio(str(path)))

### Step 2e — Hindi in your voice  (write Hindi in Devanagari script)

In [ ]:
path = cloner.speak("नमस्ते! यह मेरी अपनी आवाज़ है, जो मेरे वॉइस इंजन ने बनाई है।", language="hi")
print("Saved:", path)
display(Audio(str(path)))

### Step 2f — your own text

In [ ]:
TEXT = "Type anything you want here, and it will be spoken in my voice."
LANGUAGE = "en"                  # "en" or "hi"

path = cloner.speak(TEXT, language=LANGUAGE)
print("Saved:", path)
display(Audio(str(path)))

### Tips
- Each run sounds a little different (the model has some randomness) — run again if a result sounds odd.
- Hindi must be written in **Devanagari** (नमस्ते), not English letters ("namaste").
- It will sound *like* you, not perfectly. More clean recordings (5–10+ minutes) improve it.
- Files are in `outputs/speech/`. Colab deletes them when the session ends — download them, or copy to Drive:
  `!cp -r outputs/speech /content/drive/MyDrive/`

Next: **Step 3 — a simple engine:** `engine.speak("text", language="hi")`.